# BTF-3 — yes/no forecasts, scored by accuracy with Brier beside it

[BTF-3](https://huggingface.co/datasets/futuresearch-ai/BTF-3) (Bench to the Future 3, by
FutureSearch) holds real forecasting questions asked in late April to late May 2026 and resolved
by early July 2026. The `btf3` Benchmark serves its 1,515 **yes/no** questions from the release
that [Diverse by Reasoning](https://arxiv.org/abs/2608.24001) used, with that paper's prompt and
grading rule, so its BTF-3 results can be rerun here. Each question comes with background,
resolution criteria and a date; the model reasons, then ends with one line
`Final answer: <probability between 0 and 1>`, the probability that the answer is Yes.

**The headline is accuracy**: a probability of 0.5 or more counts as Yes. The **Brier score**
(the squared gap between the probability and what happened; lower is better, 0.25 for always
answering 0.5) is published beside it as the run metric `brier`. Read both: about two in three
questions resolved No, so always answering No already scores 0.67 on accuracy. A reply with no
readable probability is a miss and costs Brier 0.25; the run metric `no_forecast` counts them.

**Read a score with this in mind.** The model gets no research or web search, as in the paper,
so scores are not comparable with the FutureSearch leaderboard's research agents. The paper ran
every model at temperature 0; this Benchmark does not set it. The events have already happened:
a model trained after about April 2026 may simply know them. The questions are CC-BY-NC-4.0.

## 0. Before running

The next cell picks where this notebook runs: `TARGET = "local"` (the default)
uses your `screamingface up` stack, and `TARGET = "hosted"` uses
`fusion.screamingface.ai`.

**Local.** Install the local services with
`pip install "screamingface[runtime,notebook]"`, then from a terminal:

```bash
screamingface prepare btf3   # first run only: download the questions
screamingface up             # start Gateway :9105, Scoreboard :9106, and Engine :9108
screamingface status
```

Use `screamingface logs` to inspect startup failures and `screamingface down` when
finished. Stack management stays outside the notebook so **Run All** never starts or
stops local services. Working from a checkout? `just local-stack-notebooks` in
`packages/screamingface/` does every local step — assets, stack, and Jupyter — in one
command.

**Hosted.** Nothing to start: `pip install "screamingface[notebook]"` is enough, and
`sf.connect()` opens the login in your browser.

In [ ]:
import screamingface as sf

TARGET = "local"  # "local": your `screamingface up` stack · "hosted": fusion.screamingface.ai

if TARGET == "local":
    sf.configure(engine_url="http://127.0.0.1:9108", scoreboard_url="http://127.0.0.1:9106")
elif TARGET == "hosted":
    sf.configure(
        engine_url="https://fusion.screamingface.ai",
        scoreboard_url="https://leaderboard.screamingface.ai",
    )
else:
    raise ValueError(f'TARGET must be "local" or "hosted", not {TARGET!r}')

sf.connect()

## 1. Read the Benchmark's card

`btf3` was written by a contributor outside the team (origin `external`). The card carries the
pinned release, the licence and the scoring caveats.

In [ ]:
sf.benchmarks.get("btf3")

## 2. Run a few questions with one model

The prompt asks for five reasoning steps before the probability, so leave room for them. To
match the paper, add `"temperature": 0` for a model that accepts it.

In [ ]:
# No `temperature` here on purpose: several current reasoning models reject the parameter
# outright (see 13_contracteval for the provider-by-provider detail).
PARAMS = {"max_tokens": 4096}

solo = sf.Model(model="openrouter/openai/gpt-5.5", params=PARAMS)
report = sf.evaluate(solo, benchmark="btf3", limit=5)
report

## 3. Read each forecast

Each case carries the probability read from the reply, the outcome (1 = Yes), its Brier and
whether the reply ended on the probability as asked (`compliant`). A reply with no probability
shows `forecast` as None, scores 0 and costs Brier 0.25; the run's metrics count those cases as
`no_forecast`, beside `brier`.

In [ ]:
for case in report.candidates.only.cases:
    grade = case.grade
    if grade is None or case.failures:
        print(case.case_id, case.status, [failure.code for failure in case.failures])
        continue
    meta = grade.checks[0].evidence[0].metadata
    fields = [meta.get(key) for key in ("forecast", "outcome", "brier", "compliant")]
    print(case.case_id, *fields)

## 4. Before you scale up

A `limit=N` run is a smoke test, not a ranking: about two in three questions resolved No, and
five questions will not sample that evenly. One pass over all 1,515 questions is 1,515
Candidate calls per model, with no Judge. There is no spend cap in this stack, so `limit` is the
only brake.